In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [3]:
from sklearn.datasets import load_breast_cancer
import pandas as pd

data = load_breast_cancer()

df = pd.DataFrame(
    data.data,
    columns=data.feature_names
)

df["target"] = data.target

print(df.head())
print(df["target"].value_counts())

   mean radius  mean texture  mean perimeter  mean area  mean smoothness  \
0        17.99         10.38          122.80     1001.0          0.11840   
1        20.57         17.77          132.90     1326.0          0.08474   
2        19.69         21.25          130.00     1203.0          0.10960   
3        11.42         20.38           77.58      386.1          0.14250   
4        20.29         14.34          135.10     1297.0          0.10030   

   mean compactness  mean concavity  mean concave points  mean symmetry  \
0           0.27760          0.3001              0.14710         0.2419   
1           0.07864          0.0869              0.07017         0.1812   
2           0.15990          0.1974              0.12790         0.2069   
3           0.28390          0.2414              0.10520         0.2597   
4           0.13280          0.1980              0.10430         0.1809   

   mean fractal dimension  ...  worst texture  worst perimeter  worst area  \
0             

In [4]:
df = df.dropna()

In [5]:
y = df["target"]
X = df.drop(columns=["target"])

In [6]:
print(df["target"].unique())


[0 1]


# Creating The Model

In [7]:

def getsplit(X, column):
    # we will make a list that contains every possible split for this column 
    values = X[column].unique()
    if values.dtype == "object":
        return values.tolist()
    else:
        values = sorted(values)
        split = []
        for i in range(len(values)-1):
            avg = (values[i]+values[i+1])/2
            split.append(round(avg, 2))
        return split


In [8]:
def split_data(df, column, value):
    # if it is an object we will split the data : the left will take the lines where col == value and right will take the rest
    if df[column].dtype == "object":
        left = df[df[column] == value]
        right = df[df[column] != value]
    # if it is an int/float we will split the data : the left will take the lines where col < value and right will take the rest
    else:
        left = df[df[column] < value]
        right = df[df[column] >= value]
    return (left,right)

In [9]:
def mse(data, target):
    y = data[target]
    y_mean = y.mean()
    error = 0
    for y_i in y:
        error += (y_i - y_mean)**2
    return error *(1 / len(y))

In [10]:
def weighted_mse(left, right, target):
    nb_left, nb_right = len(left), len(right)
    total = nb_left + nb_right
    total_mse = (nb_left / total) * mse(left, target) + (nb_right / total) * mse(right, target)
    return total_mse

In [11]:
def best_split(df, target):
    # we intialize the variables
    best_wg, best_split, best_column = float("inf"), None, None
    X = df.drop(target, axis = 1)
    for column in X.columns:
        # for every column we will get a list of the possible splits
        splits = getsplit(X, column)
        for split in splits:
            # for every split we will split it again so that it would be left, right respecting the rules that we mentionned before
            left, right = split_data(df, column, split)
            if len(left) == 0 or len(right) == 0:
                continue
            wg = weighted_mse(left, right, target)
            # choosing the best split (the one that has the least wg)
            if wg < best_wg :
                best_wg = wg
                best_split = split
                best_column = column
    return best_wg, best_split, best_column

In [12]:
def should_stop(df, target, depth, max_depth, min_samples_split):
    if  df[target].nunique() == 1: # y contains only 1 unique value
        return True
    elif max_depth is not None and depth >= max_depth : # max depth reached
        return True
    elif len(df) < min_samples_split : # not enough samples
        return True
    return False

In [13]:
def create_leaf(df, probabilities):
    # we create a leaf and prediction is gamma with gamma / sum(residual)/sum(p*(1-p))
    sum_residual = sum(df["residual"])
    leaf_probabilities = probabilities[df.index] # we use this to make sure that the probabilities has the same index as d
    sum_probabilities = sum(leaf_probabilities * (1 - leaf_probabilities))
    gamma = sum_residual / sum_probabilities
    return {"type": "leaf" , "prediction" : gamma}
        

In [14]:
def build_tree(df, target, probabilities, depth=0, max_depth=None, min_samples_split=2):
    # reccursive function that build a tree
    if should_stop(df, target, depth, max_depth, min_samples_split):
        return create_leaf(df, probabilities)
    best_wg, best_value, best_column = best_split(df, target)
    if best_column is None:
        return create_leaf(df, probabilities)
    left, right = split_data(df, best_column, best_value)
    left_tree = build_tree(left, target, probabilities, depth+1, max_depth, min_samples_split)
    right_tree = build_tree(right, target, probabilities, depth+1, max_depth, min_samples_split)
    return {
    "type": "node",
    "column": best_column,
    "value": best_value,
    "left": left_tree,
    "right": right_tree
    }

# Gradient Boosting

In [15]:
def encode_labels(y):
    classes = sorted(y.unique())

    if len(classes) != 2:
        raise ValueError("Only binary classification is supported.")

    mapping = {
        classes[0]: 0,
        classes[1]: 1
    }

    encoded_y = y.map(mapping)

    return encoded_y, mapping

In [16]:
def decode_labels(predictions, mapping):
    reverse_mapping = {
        value: key for key, value in mapping.items()
    }

    return predictions.map(reverse_mapping)

In [17]:
def initialize_model(y):
    # predicting f_0 which is a constant value called the log(odds) or log(p/1-p)
    encoded_y, mapping = encode_labels(y)
    p = encoded_y.mean()
    # avoiding zero
    eps = 1e-15
    p = np.clip(p, eps, 1 - eps)
    log_odds = np.log(p / (1 - p))
    predictions = np.full(len(y), log_odds)
    return predictions, encoded_y, mapping

In [18]:
def pseudo_residual(y, prediction):
    return y - prediction

In [ ]:
def gradient_boosting_classification(df, target, m, learning_rate, max_depth):
    X = df.drop(target, axis=1)
    train_df = X.copy()
    y = df[target]
    predictions, encoded_y, mapping = initialize_model(y)
    initial_prediction = predictions.copy()
    trees = []
    for i in range(m):
        probabilities = sigmoid(predictions)
        train_df["residual"] = pseudo_residual(encoded_y, probabilities)
        tree = build_tree(train_df, "residual", probabilities, max_depth=max_depth)
        residual_predictions = predict_multiple_lines(tree, X)
        residual_predictions = np.array(residual_predictions)
        predictions += learning_rate * residual_predictions
        trees.append(tree)
    return initial_prediction, trees

In [ ]:
def predict_gradient_boosting(trees, initial_prediction, X, learning_rate):
    predictions = np.full(len(X), initial_prediction)

    for tree in trees:
        tree_predictions = predict_multiple_lines(tree, X)
        predictions += learning_rate * np.array(tree_predictions)

    return predictions

In [ ]:
def predict(tree, sample):
    if tree["type"] == "leaf" :
        return tree["prediction"]
    column = tree["column"]
    value = tree["value"]

    if isinstance(value, (int, float, np.number)): # we see if the value is numerical or not
        if sample[column] < value :
            return predict(tree["left"], sample) 
        else:
            return predict(tree["right"], sample)
    else:
        if sample[column] == value :
            return predict(tree["left"], sample) 
        else:
            return predict(tree["right"], sample)
        
    

In [ ]:
def predict_multiple_lines(tree, X):
    predictions = []
    for _, row in X.iterrows():
        predictions.append(predict(tree, row))
    return predictions

In [ ]:
def sigmoid(x):
    x = np.clip(x, -500, 500)
    return 1 / (1 + np.exp(-x))

In [ ]:
def accuracy(y_true, y_pred):
    correct = 0

    for true, pred in zip(y_true, y_pred):
        if true == pred:
            correct += 1

    return correct / len(y_true)

# Predicting result

In [ ]:
train = df.sample(frac=0.8, random_state=42)
test = df.drop(train.index)

In [ ]:
initial_prediction, trees = gradient_boosting_classification(df, "target", m = 5, learning_rate = 0.1, max_depth = 2)

In [42]:
X_test = test.drop("target", axis=1)
y_test_me = test["target"]
log_odds = predict_gradient_boosting( trees, initial_prediction[0], X_test, learning_rate=0.1)
probabilities = sigmoid(log_odds)
y_pred = (probabilities >= 0.5).astype(int)


In [43]:
print(accuracy(y_test_me, y_pred))

0.9824561403508771


# Creating the same model using sklearn

In [44]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score

model = GradientBoostingClassifier(
    n_estimators=5,
    learning_rate=0.1,
    max_depth=2
)

model.fit(
    train.drop("target", axis=1),
    train["target"]
)

sk_pred = model.predict(X_test)


# my model vs scikit-learn 

In [45]:
print(f" my accuracy :{accuracy(y_test_me, y_pred)} vs scikit-learn's :{accuracy_score(sk_pred, y_pred)}")

 my accuracy :0.9824561403508771 vs scikit-learn's :0.9385964912280702


# Conclusion : Mine is better 

In [ ]:
# i chose m=5 because it takes so much time to be executed and i created 2 functions that i didn't use encode_labels , decode_labels 
# because i chose the dataset after creating the model and i was thinking that i will choose a dataset that has an object type target